[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/metaflow-certified/notebooks/day-12-production-flows.ipynb#scrollTo=aa1b2c3d)

---
# Day 12 · Production Flows: Scheduling and Deployment
**certified-journeys / metaflow-certified** · Practice · Production

> **Goal for today:** Understand Metaflow's production deployment model — the `@schedule` decorator, Argo Workflows and AWS Step Functions integration, deployment command syntax, and production best practices — with a locally-simulated production workflow.


In [ ]:
%pip install -q metaflow scikit-learn pandas numpy


## Step 1 · Metaflow's Production Deployment Model

Metaflow separates **authoring** (your Python code) from **scheduling** (how/when the DAG runs). A single flow class can be:
- Run locally with `python flow.py run`
- Deployed to **Argo Workflows** (Kubernetes) with one command
- Deployed to **AWS Step Functions** with one command

The DAG structure is identical — Metaflow translates `@step` / `self.next()` / `@foreach` automatically.

| Target | Deploy command | Where it runs |
|---|---|---|
| Local | `python flow.py run` | Your machine |
| Argo Workflows | `python flow.py argo-workflows create` | Kubernetes cluster |
| AWS Step Functions | `python flow.py step-functions create` | AWS managed infra |
| Airflow (via plugin) | `python flow.py airflow create` | Airflow scheduler |


In [ ]:
# Verify metaflow is importable and check version
import metaflow
print(f"Metaflow version: {metaflow.__version__}")

# Check available deployment targets in this installation
!python -c "from metaflow.plugins import SIDECARS; print('Metaflow plugins loaded')" 2>/dev/null || echo "Plugin check skipped"
!python -m metaflow.cli --help 2>&1 | grep -E 'argo|step-func|schedule' | head -10 || echo "CLI help parsed"


**What just happened?**

- We confirmed Metaflow is installed and checked which deployment targets are registered in this environment.
- In a Colab environment (no Kubernetes or AWS credentials), the Argo and Step Functions commands are available but will fail at the connection step — we'll simulate locally instead.
- **Key insight**: the deployment commands are CLI subcommands on the flow script itself, not separate tooling — `python flow.py argo-workflows create` is all you need.


## Step 2 · The @schedule Decorator

`@schedule` adds a time-based trigger to your flow. When deployed to Argo Workflows or Step Functions, Metaflow creates the corresponding CronWorkflow (Argo) or EventBridge rule (Step Functions) automatically.

| Decorator | Meaning |
|---|---|
| `@schedule(cron='0 9 * * 1')` | Every Monday at 9 AM UTC |
| `@schedule(daily=True)` | Every day at midnight UTC |
| `@schedule(hourly=True)` | Every hour |
| `@schedule(weekly=True)` | Every Monday at midnight UTC |

The `@schedule` decorator is **ignored during local runs** — it only has effect when the flow is deployed.


In [ ]:
%%writefile scheduled_training_flow.py
from metaflow import FlowSpec, step, schedule, Parameter
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler


# @schedule defines when this flow runs in production:
#   cron='0 2 * * 1'  =  every Monday at 02:00 UTC
# Ignored during local `python flow.py run`
@schedule(cron='0 2 * * 1')
class ScheduledTrainingFlow(FlowSpec):
    """Weekly retraining flow — scheduled via @schedule for production."""

    # Parameters can be overridden at deploy time or per-trigger
    n_estimators = Parameter('n_estimators',
                             help='Number of trees in the random forest',
                             default=100)
    test_size = Parameter('test_size',
                          help='Fraction of data held out for evaluation',
                          default=0.25)

    @step
    def start(self):
        """Load data — in production this would query a feature store."""
        iris = load_iris(as_frame=True)
        df = iris.frame.rename(columns={'target': 'species'})

        X = df.drop(columns='species').values
        y = df['species'].values

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=self.test_size, random_state=42, stratify=y
        )
        scaler = StandardScaler()
        self.X_train = scaler.fit_transform(X_train)
        self.X_test  = scaler.transform(X_test)
        self.y_train = y_train
        self.y_test  = y_test
        self.scaler  = scaler

        print(f"Parameters: n_estimators={self.n_estimators}, test_size={self.test_size}")
        self.next(self.train)

    @step
    def train(self):
        """Train the model with the configured hyperparameters."""
        from sklearn.ensemble import RandomForestClassifier
        clf = RandomForestClassifier(
            n_estimators=self.n_estimators, random_state=42
        )
        clf.fit(self.X_train, self.y_train)
        preds = clf.predict(self.X_test)
        self.model    = clf
        self.accuracy = float(accuracy_score(self.y_test, preds))
        print(f"Training accuracy: {self.accuracy:.4f}")
        self.next(self.evaluate)

    @step
    def evaluate(self):
        """Quality gate: fail the run if accuracy drops below threshold."""
        ACCURACY_THRESHOLD = 0.85
        print(f"Accuracy: {self.accuracy:.4f}  (threshold: {ACCURACY_THRESHOLD})")
        if self.accuracy < ACCURACY_THRESHOLD:
            raise ValueError(
                f"Model accuracy {self.accuracy:.4f} below production threshold "
                f"{ACCURACY_THRESHOLD} — aborting deployment!"
            )
        print("Quality gate passed — model is ready for deployment.")
        self.next(self.end)

    @step
    def end(self):
        """Register the model for serving — in production this pushes to a registry."""
        print(f"Model registered. Accuracy: {self.accuracy:.4f}")
        print(f"Scaler artifact: {type(self.scaler).__name__}")
        # Production: self.model bytes → S3 → model registry (MLflow, BentoML, SageMaker)


if __name__ == '__main__':
    ScheduledTrainingFlow()


**What just happened?**

- `@schedule(cron='0 2 * * 1')` decorates the entire flow class — when deployed, Metaflow creates a weekly trigger automatically.
- **`Parameter`** makes hyperparameters overridable at runtime: `python flow.py run --n_estimators 200`.
- The `evaluate` step acts as a **quality gate** — if accuracy drops (e.g., due to data drift), the run fails and the model is never registered. This is a production safety pattern.
- `@schedule` is completely ignored during local `run` — you test locally without any changes to the code.


In [ ]:
# Run locally — @schedule is ignored, Parameters use defaults
!python scheduled_training_flow.py run


**What just happened?**

- The flow ran locally with default parameters — `@schedule` had no effect.
- The quality gate passed because iris classification accuracy easily exceeds 0.85 with a RandomForest.
- **Try overriding a parameter**: `!python scheduled_training_flow.py run --n_estimators 5` — fewer trees may drop accuracy below the threshold.


## Step 3 · Argo Workflows Integration

Argo Workflows is a Kubernetes-native DAG scheduler. Metaflow translates your `FlowSpec` into an Argo `WorkflowTemplate` automatically.

### Deploy command syntax

```bash
# Create / update the WorkflowTemplate in your connected K8s cluster
python flow.py argo-workflows create

# Trigger a one-off run (without waiting for the schedule)
python flow.py argo-workflows trigger

# List existing WorkflowTemplates for this flow
python flow.py argo-workflows list

# Terminate a running workflow
python flow.py argo-workflows terminate --run-id <id>
```

### What Metaflow generates for Argo

| Metaflow concept | Argo equivalent |
|---|---|
| `FlowSpec` class | `WorkflowTemplate` |
| `@step` method | Argo `Template` (step/DAG node) |
| `@foreach` | `withItems` or `withParam` |
| `@schedule(cron=...)` | `CronWorkflow` |
| `@resources(cpu=2, memory=4096)` | Pod resource requests/limits |
| `@retry(times=3)` | `retryStrategy.limit: 3` |


In [ ]:
# Show the full Argo deployment command syntax (will error without K8s — expected)
!python scheduled_training_flow.py argo-workflows --help 2>&1 | head -40 || echo "Argo plugin not configured in this environment (expected in Colab)"


**What just happened?**

- In Colab (no K8s cluster), the Argo plugin is not configured — this is expected.
- **In a real environment**: run `python flow.py argo-workflows create` from a machine with `kubectl` configured and Argo Workflows installed in your cluster.
- Metaflow reads your cluster config from `~/.metaflowconfig` (set via `metaflow configure kubernetes`).
- The `create` command is idempotent — re-running it updates the existing WorkflowTemplate.


## Step 4 · AWS Step Functions Integration

AWS Step Functions is a serverless DAG orchestrator. Metaflow generates a State Machine definition automatically — no JSON or ASL to write by hand.

### Deploy command syntax

```bash
# Create or update the State Machine in AWS
python flow.py step-functions create

# Trigger a one-off execution
python flow.py step-functions trigger

# View execution status
python flow.py step-functions list-runs
```

### Required AWS setup (one-time)

```bash
# Configure Metaflow for AWS (sets up S3, batch, Step Functions)
metaflow configure aws

# What gets configured:
#  METAFLOW_DATASTORE_SYSROOT_S3  = s3://your-bucket/metaflow
#  METAFLOW_DEFAULT_DATASTORE     = s3
#  METAFLOW_DEFAULT_METADATA      = service  (or local)
#  AWS_DEFAULT_REGION             = us-east-1
```

### IAM permissions needed

| Service | Required permissions |
|---|---|
| S3 | `GetObject`, `PutObject` on metaflow bucket |
| Step Functions | `CreateStateMachine`, `StartExecution`, `DescribeExecution` |
| AWS Batch | `SubmitJob`, `DescribeJobs` (if using `@batch` decorator) |
| CloudWatch | `PutLogEvents` (for step logs) |


In [ ]:
%%writefile production_flow_with_decorators.py
# This file demonstrates the SYNTAX of production decorators.
# It is runnable locally — cloud decorators (@batch, @kubernetes) are
# ignored unless Metaflow is configured for that backend.

from metaflow import FlowSpec, step, schedule, retry, timeout, Parameter

# In production these would be active; locally they are no-ops
try:
    from metaflow import batch, kubernetes
    HAS_CLOUD = True
except ImportError:
    # Define dummy decorators so the file is importable anywhere
    def batch(**kw):    return lambda f: f
    def kubernetes(**kw): return lambda f: f
    HAS_CLOUD = False


@schedule(cron='0 6 * * *')  # daily at 06:00 UTC
class ProductionTrainingFlow(FlowSpec):
    """
    Production-grade training flow demonstrating:
      - @schedule for daily retraining
      - @retry for transient failure recovery
      - @timeout to prevent runaway steps
      - @batch / @kubernetes (syntax shown, not executed without cloud config)
    """

    model_version = Parameter('model_version',
                              help='Semantic version tag for this model',
                              default='1.0.0')

    @retry(times=3, minutes_between_retries=1)
    @timeout(minutes=30)
    # @batch(cpu=4, memory=16000, image='python:3.10')  # AWS Batch (comment in for prod)
    # @kubernetes(cpu=2, memory=8192)                   # K8s (comment in for prod)
    @step
    def start(self):
        """
        Data ingestion step.
        @retry:   retries up to 3 times on transient errors (S3 throttling, network blips)
        @timeout: kills the step if it runs for more than 30 minutes
        @batch:   runs on AWS Batch with 4 CPUs, 16 GB RAM (when configured)
        """
        import pandas as pd
        from sklearn.datasets import load_iris
        iris = load_iris(as_frame=True)
        self.raw_df = iris.frame.rename(columns={'target': 'species'})
        print(f"Ingested {len(self.raw_df)} rows | model_version={self.model_version}")
        self.next(self.train)

    @retry(times=2)
    @timeout(minutes=60)
    @step
    def train(self):
        """Training step — isolated retry budget from the data step."""
        import numpy as np
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.model_selection import train_test_split
        from sklearn.metrics import accuracy_score
        from sklearn.preprocessing import StandardScaler

        df = self.raw_df
        X = df.drop(columns='species').values
        y = df['species'].values

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s  = scaler.transform(X_test)

        clf = RandomForestClassifier(n_estimators=100, random_state=42)
        clf.fit(X_train_s, y_train)
        preds = clf.predict(X_test_s)

        self.model    = clf
        self.scaler   = scaler
        self.accuracy = float(accuracy_score(y_test, preds))
        print(f"Trained model | accuracy={self.accuracy:.4f}")
        self.next(self.end)

    @step
    def end(self):
        print(f"Production run complete")
        print(f"  model_version : {self.model_version}")
        print(f"  accuracy      : {self.accuracy:.4f}")
        print("  Deployment:")
        print("    Argo  : python production_flow_with_decorators.py argo-workflows create")
        print("    AWS SF: python production_flow_with_decorators.py step-functions create")


if __name__ == '__main__':
    ProductionTrainingFlow()


In [ ]:
# Run locally — cloud decorators (@batch, @kubernetes) are no-ops without config
!python production_flow_with_decorators.py run


**What just happened?**

- The flow ran with `@retry`, `@timeout`, and `@schedule` applied but ignored in local mode — exactly as designed.
- **`@retry(times=3)`** means: if `start` raises an exception, Metaflow waits `minutes_between_retries` and tries again up to 3 more times before marking the step as failed.
- **`@timeout(minutes=30)`** sets a wall-clock limit per step — essential for preventing runaway training jobs from consuming cloud resources indefinitely.
- **`@batch` / `@kubernetes`** are commented out — uncomment them when deploying to a configured cloud environment. The local run uses your current Python process regardless.


## Step 5 · Production Deployment Best Practices

A checklist for moving a Metaflow flow from development to production:

| Practice | Why it matters |
|---|---|
| **Pin image versions** | `@batch(image='myecr.io/ml:2.1.0')` — don't use `latest` |
| **Add `@retry` to all I/O steps** | Network and object-store errors are transient |
| **Add `@timeout` to all steps** | Prevents runaway training from billing you indefinitely |
| **Use `@environment(vars={...})`** | Pass secrets via env vars, not code |
| **Quality gate in `evaluate` step** | Block bad models from reaching the registry |
| **Store model version in Parameter** | Track which artifact belongs to which deployment |
| **Use `@project` for namespacing** | Isolates dev/staging/prod runs in the same Metaflow service |
| **Never run production flows locally** | Use `trigger` to start a run — keeps lineage in the service |


In [ ]:
%%writefile production_best_practices.py
"""
Demonstrates @project for namespace isolation and @environment for secret injection.
Runnable locally — cloud-specific decorators shown but commented out.
"""
from metaflow import FlowSpec, step, Parameter

try:
    from metaflow import project, environment
except ImportError:
    def project(**kw): return lambda cls: cls
    def environment(**kw): return lambda f: f


# @project groups flows into an isolated namespace.
# Runs in 'dev' branch don't pollute 'prod' artifact lineage.
# Usage: python flow.py run --branch dev  OR  --branch prod
@project(name='ml_platform')
class BestPracticesFlow(FlowSpec):
    """
    Demonstrates @project namespace isolation and @environment for secret injection.
    """

    # Parameters let you change behaviour at trigger time without editing code
    env_name = Parameter('env', help='Deployment environment', default='dev')

    # @environment injects env vars into the step — use for secrets (API keys, DB passwords)
    # Never hardcode secrets in flow code!
    # @environment(vars={'DB_PASSWORD': os.environ.get('DB_PASSWORD', '')})
    @step
    def start(self):
        print(f"Running in environment: {self.env_name}")
        # In production:
        #   - fetch feature data from a feature store
        #   - use @environment(vars={'FEATURE_STORE_API_KEY': os.environ['FS_KEY']})
        #   - never print or log secret values
        self.data_version = '2024-06-06'  # would come from a feature store in prod
        self.next(self.process)

    @step
    def process(self):
        import numpy as np
        # Simulate processing
        self.processed = True
        print(f"Processed data_version={self.data_version} in env={self.env_name}")
        self.next(self.end)

    @step
    def end(self):
        print("Run complete.")
        # Production deployment commands (run from a CI/CD pipeline):
        # Deploy to Argo Workflows:
        #   python best_practices_flow.py argo-workflows create
        #
        # Deploy to AWS Step Functions:
        #   python best_practices_flow.py step-functions create
        #
        # Trigger a one-off run in production:
        #   python best_practices_flow.py argo-workflows trigger --env prod


if __name__ == '__main__':
    BestPracticesFlow()


In [ ]:
!python production_best_practices.py run --env dev


**What just happened?**

- `@project(name='ml_platform')` creates a namespace — runs under different `--branch` values (dev/staging/prod) are isolated even if they share a Metaflow service.
- **`Parameter('env')`** makes the environment name overridable at trigger time without touching code — your CD pipeline passes `--env prod`.
- `@environment(vars={...})` would inject secrets as environment variables into the step container — the secret value is never written into the flow file or the artifact store.
- This pattern separates **what the flow does** (code) from **where it runs** (decorators) and **what secrets it uses** (environment vars).


## Step 6 · Simulating a Production Deployment Workflow

In a real CI/CD pipeline, the deployment workflow is:

```
git push → CI runs tests → CD deploys to staging → promote to prod
```

In Metaflow terms:

```bash
# 1. Run tests locally
python flow.py run --env dev

# 2. Deploy to staging (Argo)
python flow.py argo-workflows create --branch staging

# 3. Trigger staging run
python flow.py argo-workflows trigger --env staging

# 4. After review, deploy to production
python flow.py argo-workflows create --branch prod

# 5. Production runs on @schedule automatically; trigger ad-hoc:
python flow.py argo-workflows trigger --env prod
```

We simulate this pattern locally by running with different `--env` parameter values.


In [ ]:
# Simulate the staging → production promotion pattern locally
import subprocess

environments = ['dev', 'staging', 'prod']

for env in environments:
    print(f"\n{'='*50}")
    print(f"Simulating deployment: env={env}")
    print(f"{'='*50}")
    result = subprocess.run(
        ['python', 'production_best_practices.py', 'run', '--env', env],
        capture_output=True, text=True
    )
    # Show the last few lines of output for each run
    output_lines = (result.stdout + result.stderr).strip().split('\n')
    for line in output_lines[-6:]:
        print(f"  {line}")


**What just happened?**

- We simulated three separate runs (dev → staging → prod) using different `--env` parameter values.
- Each run is a separate entry in the Metaflow run history — independently queryable and comparable via the Client API.
- In a real deployment, `dev` and `staging` run on the Kubernetes cluster but in separate `@project` branches, while `prod` has the `@schedule` trigger enabled.
- **The flow code is identical across all environments** — only the parameter value and cloud config differ. This is the key advantage of Metaflow's deployment model.


In [ ]:
# Review all simulated runs via the Client API
from metaflow import Flow
import pandas as pd

flow = Flow('BestPracticesFlow')
rows = []
for run in flow.runs():
    if run.successful:
        try:
            d = run['end'].task.data
            rows.append({
                'run_id':       run.id,
                'env':          d.env_name,
                'data_version': d.data_version,
                'finished_at':  run.finished_at,
            })
        except Exception:
            pass

if rows:
    df = pd.DataFrame(rows)
    print("Deployment simulation run history:")
    print(df.to_string(index=False))
else:
    print("No completed runs found — run the simulation cell above first.")


**What just happened?**

- We iterated all successful runs and extracted the `env_name` and `data_version` artifacts — a simple deployment audit trail.
- **Every run is immutable**: you can always trace back which code, data version, and parameters produced any given model artifact.
- In production, you'd add `model_accuracy`, `model_registry_url`, and `deployed_at` to this table — giving you a complete model lineage dashboard without a separate MLOps tool.


In [ ]:
# Challenge: Build a production-ready scheduled retraining flow
# Your solution here
#
# Write a flow 'ProductionChallenge' that:
# 1. Uses @schedule(cron='0 3 * * 0')  — every Sunday at 03:00 UTC
# 2. Has a Parameter 'min_accuracy' with default 0.90
# 3. Has 4 steps: start, train, evaluate, end
#    - start: load diabetes dataset from sklearn, train/test split
#    - train: fit a RandomForestRegressor (n_estimators=100)
#              store self.r2_score = r2_score(y_test, preds)
#    - evaluate: quality gate using self.min_accuracy as the R² threshold
#    - end: print r2_score and confirm deployment commands
# 4. Add @retry(times=2) to start and train steps
# 5. Run locally and verify the quality gate fires at the default threshold
#
# Scaffold:
# from metaflow import FlowSpec, step, schedule, retry, Parameter
# from sklearn.datasets import load_diabetes
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.metrics import r2_score
#
# @schedule(cron='0 3 * * 0')
# class ProductionChallenge(FlowSpec):
#     min_accuracy = Parameter('min_accuracy', default=0.90)
#     ...


---
## Day 12 key concepts recap

| Concept | What to remember |
|---|---|
| `@schedule(cron=...)` | Adds a time-based trigger — ignored locally, active when deployed |
| `argo-workflows create` | Deploys the flow as an Argo WorkflowTemplate in one command |
| `step-functions create` | Deploys the flow as an AWS Step Functions State Machine |
| `@retry(times=N)` | Per-step retry budget — protects I/O steps from transient failures |
| `@timeout(minutes=N)` | Wall-clock limit per step — prevents runaway cloud billing |
| `@environment(vars={...})` | Inject secrets as env vars — never hardcode secrets in flow code |
| `@project(name=...)` | Namespace isolation — dev/staging/prod branches share one flow |
| Quality gate pattern | `raise ValueError(...)` in an evaluate step blocks bad models |
| `argo-workflows trigger` | Kick off a one-off run without waiting for the schedule |

> **Tip:** Deploy to Argo or Step Functions with a single command — Metaflow handles the DAG translation automatically.

---
## What's next
**Day 13** → Advanced Metaflow: `@conda` for environment isolation, `@resources` for GPU steps, and integrating Metaflow with a model registry for full MLOps pipelines.

Mark Day 12 complete in your [tracker](../index.html).
